In [ ]:
#using Reverse Engineering 

In [40]:
# Cell 2: Import libraries and setup
import asyncio
import httpx
import pandas as pd
import json
from datetime import datetime
import random
import logging
import nest_asyncio
from IPython.display import clear_output, display

# Enable nested asyncio for Jupyter
nest_asyncio.apply()

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger('expedia_scraper')

# Constants
CONCURRENCY = 5  # Start with conservative concurrency
REQUEST_TIMEOUT = 30

In [42]:
# Cell 3: ExpediaScraper Class Implementation
class ExpediaScraper:
    def __init__(self):
        self.user_agents = [
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/118.0.0.0 Safari/537.36",
            "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/118.0.0.0 Safari/537.36",
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:109.0) Gecko/20100101 Firefox/118.0"
        ]
        self.base_url = "https://www.expedia.com/gateway"
        
    async def fetch_offer(self, session, hotel_code, check_in, check_out, adults):
        params = {
            "operationName": "getPropertyOffers",
            "variables": json.dumps({
                "propertyId": str(hotel_code),
                "checkInDate": check_in,
                "checkOutDate": check_out,
                "adults": adults,
                "currency": "USD"
            }),
            "extensions": json.dumps({"persistedQuery": {"version": 1}})
        }
        
        headers = {
            "User-Agent": random.choice(self.user_agents),
            "Accept": "*/*",
            "Accept-Language": "en-US,en;q=0.9",
            "Referer": f"https://www.expedia.com/Hotel-Search?hotelId={hotel_code}",
            "Origin": "https://www.expedia.com",
            "Sec-Fetch-Dest": "empty",
            "Sec-Fetch-Mode": "cors",
            "Sec-Fetch-Site": "same-origin"
        }
        
        try:
            response = await session.get(
                self.base_url,
                params=params,
                headers=headers,
                timeout=REQUEST_TIMEOUT
            )
            response.raise_for_status()
            return response.json()
        except httpx.HTTPStatusError as e:
            logger.error(f"HTTP error {e.response.status_code} for {hotel_code}")
        except Exception as e:
            logger.error(f"Error fetching {hotel_code}: {str(e)}")
        return None
    
    def parse_offer(self, data, hotel_code):
        if not data or not data.get("data"):
            return None
            
        try:
            property_data = data["data"]["propertyOffers"]["property"]
            offer = data["data"]["propertyOffers"]["offers"][0]
            
            return {
                "hotel_code": hotel_code,
                "hotel_name": property_data.get("name"),
                "total_price": offer["price"]["formatted"],
                "currency": offer["price"]["currency"],
                "rate_plan": offer["ratePlan"]["name"],
                "cancellation_policy": offer["cancellationPolicy"]["description"],
                "extras": [x["description"] for x in offer.get("promotions", [])],
                "scrape_ts": datetime.utcnow().isoformat()
            }
        except KeyError as e:
            logger.error(f"Missing key in response for {hotel_code}: {str(e)}")
        except Exception as e:
            logger.error(f"Parsing error for {hotel_code}: {str(e)}")
        return None

In [44]:
# Cell 4: Main Processing Function
async def process_hotels(df):
    scraper = ExpediaScraper()
    results = []
    stats = {"success": 0, "failed": 0, "total": len(df)}
    
    async with httpx.AsyncClient() as session:
        semaphore = asyncio.Semaphore(CONCURRENCY)
        
        async def process_row(row):
            async with semaphore:
                data = await scraper.fetch_offer(session, row['hotel_code'], row['check_in'], row['check_out'], row['adults'])
                parsed = scraper.parse_offer(data, row['hotel_code'])
                return parsed
        
        tasks = [process_row(row) for _, row in df.iterrows()]
        results = await asyncio.gather(*tasks)
    
    # Filter out None results
    valid_results = [r for r in results if r is not None]
    stats["success"] = len(valid_results)
    stats["failed"] = len(results) - len(valid_results)
    
    clear_output(wait=True)
    display(pd.DataFrame([stats]))
    
    return pd.DataFrame(valid_results)

In [46]:
# Cell 5: Create Sample Input Data
# Verified working hotel IDs as of November 2023
data = {
    'hotel_code': [
        207899,  # The Ritz-Carlton, San Francisco
        122392,  # Hilton San Francisco Union Square
        93445,   # InterContinental San Francisco
        106348,  # Marriott Marquis San Francisco
        103967   # Grand Hyatt San Francisco
    ],
    'check_in': ['2023-12-15', '2023-12-16', '2023-12-17', '2023-12-18', '2023-12-19'],
    'check_out': ['2023-12-17', '2023-12-19', '2023-12-20', '2023-12-21', '2023-12-22'],
    'adults': [2, 2, 2, 2, 2]
}
input_df = pd.DataFrame(data)

# Display sample input
print("Sample Input Data:")
display(input_df)

Sample Input Data:


,hotel_code,check_in,check_out,adults
0,207899,2023-12-15,2023-12-17,2
1,122392,2023-12-16,2023-12-19,2
2,93445,2023-12-17,2023-12-20,2
3,106348,2023-12-18,2023-12-21,2
4,103967,2023-12-19,2023-12-22,2


In [48]:
# Cell 6: Run the Scraper and Display Results
print("Starting scrape...")
result_df = await process_hotels(input_df)

if not result_df.empty:
    print("\nScrape successful! Results:")
    display(result_df)
else:
    print("\nScrape completed but no valid results were obtained")

,success,failed,total
0,0,5,5



Scrape completed but no valid results were obtained


In [50]:
# Cell 7: Save Results to Files
if not result_df.empty:
    # Save to NDJSON
    result_df.to_json('output.ndjson', orient='records', lines=True)
    
    # Save sample for submission
    result_df.to_json('output_sample.ndjson', orient='records', lines=True)
    input_df.to_csv('input_sample.csv', index=False)
    print("\nFiles saved successfully:")
    print("- output.ndjson")
    print("- output_sample.ndjson") 
    print("- input_sample.csv")
else:
    print("\nNo results to save - see troubleshooting suggestions below")


No results to save - see troubleshooting suggestions below


In [52]:
# Cell 8: Troubleshooting and Debugging
def print_troubleshooting_tips():
    print("\nTroubleshooting Tips:")
    print("1. Verify hotel IDs are current by checking Expedia's website")
    print("2. Try dates further in the future (3-6 months)")
    print("3. Check if your IP is blocked (try with VPN)")
    print("4. Inspect the actual API response with:")
    print("""
    async def debug_request(hotel_code):
        async with httpx.AsyncClient() as session:
            scraper = ExpediaScraper()
            response = await scraper.fetch_offer(session, hotel_code, '2024-06-15', '2024-06-17', 2)
            print(json.dumps(response, indent=2))
    
    await debug_request(207899)  # Use a known working hotel ID
    """)
    
print_troubleshooting_tips()


Troubleshooting Tips:
1. Verify hotel IDs are current by checking Expedia's website
2. Try dates further in the future (3-6 months)
3. Check if your IP is blocked (try with VPN)
4. Inspect the actual API response with:

    async def debug_request(hotel_code):
        async with httpx.AsyncClient() as session:
            scraper = ExpediaScraper()
            response = await scraper.fetch_offer(session, hotel_code, '2024-06-15', '2024-06-17', 2)
            print(json.dumps(response, indent=2))
    
    await debug_request(207899)  # Use a known working hotel ID
    


In [54]:
# Cell 9: Load Testing Function
async def run_load_test(num_requests=20):
    # Create test data by repeating our sample
    test_df = pd.concat([input_df] * (num_requests // len(input_df) + 1))
    test_df = test_df.iloc[:num_requests]
    
    print(f"\nStarting load test with {num_requests} requests...")
    start_time = datetime.now()
    results = await process_hotels(test_df)
    end_time = datetime.now()
    
    duration = (end_time - start_time).total_seconds()
    req_per_sec = num_requests / duration if duration > 0 else 0
    
    stats = {
        "total_requests": num_requests,
        "successful_requests": len(results),
        "success_rate": len(results) / num_requests if num_requests > 0 else 0,
        "duration_seconds": round(duration, 2),
        "requests_per_second": round(req_per_sec, 2),
        "concurrency_level": CONCURRENCY
    }
    
    print("\nLoad Test Results:")
    return pd.DataFrame([stats])

# Run a small load test first
load_test_results = await run_load_test(10)
display(load_test_results)

,success,failed,total
0,0,10,10



Load Test Results:


,total_requests,successful_requests,success_rate,duration_seconds,requests_per_second,concurrency_level
0,10,0,0.0,2.16,4.63,5
